# The Web Search Tool

Claude's built-in **web search** is a **server-side** tool: unlike the custom tools and the text editor, **you don't implement anything** — Claude runs the whole search loop on Anthropic's infrastructure. You just declare a small schema and read the results back.

> ⚠️ **Prerequisite:** your organization must **enable Web Search** in the console first — `https://console.anthropic.com/settings/privacy`. If it's not enabled, the call errors. (It's also a **billed** server tool.)

**Schema:**
```python
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
}
```
`max_uses` caps how many searches Claude may run in one turn (it may do follow-up searches based on early results).

> **Adaptations:**
> - **Model → `claude-haiku-4-5-20251001`** (the reference's `claude-sonnet-4-5` + hard-coded `temperature=1.0` would 400 on flagships).
> - **Tool version.** `web_search_20250305` is correct here — the newest flagships (Opus 4.8/4.7, Sonnet 5, Sonnet 4.6) use the updated `web_search_20260209`, but Haiku 4.5 stays on the basic variant, which matches the course.

## Setup

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from anthropic import Anthropic
from anthropic.types import Message

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })


def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## A renderer for the response blocks

A web-search response is multi-block. This helper walks them so we can *see* what Claude searched and which sources it cited:

- **`text`** — Claude's prose (may carry inline **citations**: title, url, quoted text)
- **`server_tool_use`** — the exact search **query** Claude issued (Anthropic ran it)
- **`web_search_tool_result`** — the list of results (title + url) Claude got back

In [2]:
def render_web_response(message):
    for block in message.content:
        if block.type == "text":
            print(block.text)
            for c in (getattr(block, "citations", None) or []):
                title = getattr(c, "title", "") or ""
                url = getattr(c, "url", "") or ""
                print(f"   ↳ cite: {title} — {url}")

        elif block.type == "server_tool_use":
            query = block.input.get("query") if isinstance(block.input, dict) else block.input
            print(f'\n>>> web_search query: {query}')

        elif block.type == "web_search_tool_result":
            content = block.content
            if isinstance(content, list):
                print(">>> results:")
                for r in content:
                    title = getattr(r, "title", "") or ""
                    url = getattr(r, "url", "") or ""
                    print(f"   - {title} — {url}")
            else:
                print(">>> web_search result:", content)
        print()

## Example 1 — open web search

A question that needs current information. Claude decides to search, issues a query (server-side), and answers with citations.

In [3]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
}

messages = []
add_user_message(messages, "What are the latest models Anthropic has released, and when?")

response = chat(messages, tools=[web_search_schema])
render_web_response(response)


>>> web_search query: Anthropic latest model releases 2026

>>> results:
   - Anthropic Release Notes - July 2026 Latest Updates - Releasebot — https://releasebot.io/updates/anthropic
   - Anthropic Claude Model Release Timeline - Model Family Tree, Capability Evolution, and Platform Availability | hidekazu-konishi.com — https://hidekazu-konishi.com/entry/anthropic_claude_model_release_timeline.html
   - GitHub - jqueryscript/anthropic-claude-timeline: A public timeline of major Anthropic Claude model releases, product updates, and developer platform milestones. · GitHub — https://github.com/jqueryscript/anthropic-claude-timeline
   - Anthropic's latest model release in 2026, and how a wrapper hands it to you the day it ships — https://fazm.ai/t/anthropic-latest-model-release-2026
   - Introducing Claude Opus 4.8 \ Anthropic — https://www.anthropic.com/news/claude-opus-4-8
   - Project Glasswing: Securing critical software for the AI era \ Anthropic — https://www.anthropic.com/glasswi

Raw response object, to see the actual block types (`ServerToolUseBlock`, `WebSearchToolResultBlock`, citation objects):

In [4]:
response

Message(id='msg_011CcxVcqAqfxsZXfNiT5R2y', container=None, content=[ServerToolUseBlock(id='srvtoolu_01JKF6wmpxHWCvjJ345FwKnK', caller=None, input={'query': 'Anthropic latest model releases 2026'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EpohCioIERgCIiQ5YTQ2MGU5ZS1lYmE2LTQ1NTUtODBmMi0wYTE4MTgzZWUwODESDHAXvOfIeG+wAhI41BoMLI2sXc3Upqit52qBIjBPc/hIPkFxmGJTpGvH7A5kZfVm6Xu7PBMaS45UZEWajBoIr7lty0SMatoex992SDUqnSDL6yTAig3XYW1rv2zwk1OgKm53pX8/ymY6yYoQpPsnQdwhz7Cu8eEVuQ1k/19SeHc1elgXxUFGQXW1YE4Hz5xCgBUKSy0bucemO6xTLcWGO9SEL/KYHvJGcoYEYTjk9/Xq2OR2rwaME9l0PCvNI9ZeVy5VuJexDZGqsoz5haYdl/AvKwFbA5Xqe+ceLc+qoZGhQNE5UIZs4OcRTNwkGivmbSTPogdrOP3Ptk756GTD7pjwrqsfe7krfOp7/paBukL4hQPjc+8ICMXO7vAPWFQpqhTXKXult40X4U42LuOY2mzsMdsOh47HtmMmA1G4ndjlnhsJw+4P1zi9RFoyReGDWrc0TuBWzbpQm4IGApqqXU5eNq7gANxEwM33lPeZpb9WqBKh8VsJVMBvxufCbcdUlklBE8651v2neLDzjNkIjAIcpK95nVGu3SiPBEwq7tw99+R1NWCeGkgW6wyEqz9HH0WdPfDxG

## Example 2 — restrict to authoritative domains

`allowed_domains` limits searches to sources you trust. For medical/exercise advice, pinning to `nih.gov` yields evidence-based results instead of random blogs. (There's also `blocked_domains` for the inverse.)

In [5]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

messages = []
add_user_message(messages, "What's the best exercise for gaining leg muscle?")

response = chat(messages, tools=[web_search_schema])
render_web_response(response)

The best exercise for gaining leg muscle depends on your goals and fitness level, but there are several highly effective options:

**Squats** are often considered the gold standard for leg muscle building. They work multiple leg muscles simultaneously, including the quadriceps, hamstrings, and glutes, making them very efficient for overall leg development.

**Deadlifts** are another excellent compound exercise that targets the hamstrings, glutes, and lower back, providing comprehensive leg muscle stimulation.

**Lunges** are great for building leg strength and muscle, as they work each leg individually and help correct muscle imbalances.

**Leg press** machines are effective for isolating and building leg muscles with controlled movement.

For optimal results, consider:
- **Combining exercises**: Using multiple compound movements and isolation exercises creates a well-rounded leg training program
- **Progressive overload**: Gradually increasing weight or reps over time is crucial for m

## Rendering & when to use

The block structure is built for UI: render **text** as content, list **web_search_tool_result** sources at the top, and show **citations** inline (domain, title, url, quoted text) — which builds trust by making it clear which claim came from which source.

**Best for:** current events, specialized info outside training data, fact-checking against authoritative sources, and research needing up-to-date data. Just include the schema — Claude decides when a search actually helps.

**Server-side vs. custom tools:** web search needs no `run_tool` / `run_tools` / loop on your side — Claude executes the searches and returns finished results with citations in one response. Contrast the reminder tools (you ran every call) and the text editor (you implemented every command).

That's the last piece of the tool-use section — next is the **Quiz on tool use with Claude**.